# `syft-crypto-core` CLI Quick Start

**Time Required**: 5 minutes

**Goal**: Verify that syft-crypto-core CLI utility is installed and working correctly.

In this notebook, we will:
1. Build the CLI
2. Generate an identity key
3. Encrypt a message
4. Decrypt it back

✅ **If you see "Hello World" at the end, everything works!**

---

## Prerequisites

- Rust toolchain installed
- [`just`](https://github.com/casey/just) command runner installed
- Jupyter kernel for Rust notebooks installed (with `just install-jupyter`)
- Run this notebook with the installed Rust Jupyter kernel
---

## Step 1: Build the CLI

First, let's compile the syft-crypto-cli (`syc`) binary in release mode.

In [ ]:
!cargo build --release --package syft-crypto-cli

Check that `syft` works 

In [ ]:
!../target/release/syc -h

In [ ]:
!ln -sf ../target/release/syc ./syc

## Step 2: Setting Up Your Test Environment

Use the automated setup command:

```bash
just init-sandbox
```
which creates the `sandbox` directory with the following structure to test e2e encryption for
two identities (`alice` and `bob`):
```
sandbox/
├── alice/
│   ├── .syc/                          ← Alice's vault (PRIVATE)
│   │   ├── config/datasite.json       ← Config: where files live
│   │   ├── keys/alice@example.org.key ← Private keys (KEEP SECRET!)
│   │   └── bundles/bob@...json        ← Bob's cached public key (trusted)
│   ├── datasites/                     ← Encrypted files (PUBLIC)
│   │   └── alice@example.org/
│   │       └── public/crypto/did.json ← Alice's public key
│   └── unencrypted/                   ← Plaintext workspace (LOCAL)
│       └── alice@example.org/
│           └── shared/bob@.../        ← Files to encrypt for Bob
└── bob/ (same structure to Alice)
    ├── .syc/
    ├── datasites/ 
    └── unencrypted/
```

**Key Insight:** 
- 🔐 **`.syc/keys/`** = Your secret keys (never share!)
- 🌍 **`datasites/`** = Encrypted files (safe to share publicly)
- 📝 **`unencrypted/`** = Your plaintext workspace (before/after encryption)

In [ ]:
!just clean-sandbox

In [ ]:
!just init-sandbox

## Step 3: Generate Keys

The last step (sandboxing) already generated the keys automatically for us.  
But let's create them again using `syc key generate` for each party to learn how
to create these keys

- 🔑 Recovery Key (private only)             - Your Master Backup
- ✍️ Ed25519 Identity Key (public + private) - Your digital signature
- 🔐 X25519 Signed Prekey (public + private) - Classic Diffie-Hellman keys for keys exchange
- 🛡️ Kyber1024 PQ Prekey ((public + private) - Post-quantum secure key exchange

**Expected output:**
```
✅ Keys generated for alice@example.org
📝 Recovery key (SAVE THIS!): XXXX-XXXX-XXXX-XXXX-XXXX-XXXX-XXXX-XXXX
💾 Private keys saved to: sandbox/alice/.syc/keys/test@example.org.key
🌍 Public bundle saved to: sandbox/datasites/alice@example.org/public/did.json
```

In [ ]:
!./syc key generate --overwrite --vault ../sandbox/alice/.syc --identity alice@example.org --bundle-out alice@example.org/public/crypto/did.json

In [ ]:
!./syc key generate --overwrite --vault ../sandbox/bob/.syc --identity bob@example.org --bundle-out bob@example.org/public/crypto/did.json 

## Step 4: Parties (Alice and Bob) Share Public Keys with Each Other

Alice shares her keys bundle with Bob

In [ ]:
!mkdir -p ../sandbox/bob/datasites/alice@example.org/public/crypto
!cp ../sandbox/alice/datasites/alice@example.org/public/crypto/did.json ../sandbox/bob/datasites/alice@example.org/public/crypto/did.json

which will copy alice's `did.json` into Bob's datasites
```
sandbox/
└── bob/
    ├── .syc/                           ← Bob's vault (private keys)
    ├── datasites/                      ← Encrypted files (public/shareable)
    │   ├── bob@example.org/
    │   └── alice@example.org/
    │       └── public/
    │           └── crypto/
    │               └── did.json       ← Alice's public bundle (shared)
```

Bob import Alice's bundle into his vault

In [ ]:
!./syc --vault ../sandbox/bob/.syc key import --bundle alice@example.org/public/crypto/did.json --expected-identity alice@example.org

which saves alice's bundle into byb's `.syc/bundles`
```
sandbox/
└── bob/
    ├── .syc/                           
    │   ├── config/
    │   ├── keys/
    │   └── bundles/
    │       └── alice@example.org.json ← ✅ Alice's cached public keys (TOFU)
```

Similarly, Bob also shares his bundle with Alice

In [ ]:
!mkdir -p ../sandbox/alice/datasites/bob@example.org/public/crypto
!cp ../sandbox/bob/datasites/bob@example.org/public/crypto/did.json ../sandbox/alice/datasites/bob@example.org/public/crypto/did.json

In [ ]:
!./syc --vault ../sandbox/alice/.syc key import --bundle bob@example.org/public/crypto/did.json --expected-identity bob@example.org

```
sandbox/
└── alice/
    ├── .syc/                           
    │   └── bundles/
    │       └── bob@example.org.json ← ✅ Bob's cached public keys (TOFU)
```

## Step 4: Alice Encrypts a Message & Sends to Bob

Now, Alice encrypts a message ("Hello Bob") and sends it to Bob

First, let's create the plaintext

In [ ]:
!echo "Hello Bob" > ../sandbox/alice/unencrypted/alice@example.org/shared/bob@example.org/files/message.txt

```
alice/
  ├── .syc/                                          
  ├── datasites/                                     
  └── unencrypted/                                   
      └── alice@example.org/
          ├── public/                               
          └── shared/bob@example.org/files/
              └── message.txt                       ← ✅ Plaintext message TO Bob
```

Then, encrypt with relative path, with `bob@example.org` as the recipient

In [ ]:
!./syc --vault ../sandbox/alice/.syc file encrypt --relative alice@example.org/shared/bob@example.org/files/message.txt --recipient bob@example.org

The ciphertext now lives at `sandbox/alice/datasites/alice@example.org/shared/bob@example.org/files/message.txt`

```
sandbox/
└── alice/
    ├── .syc/                                    
    ├── datasites/                               
    │   └── alice@example.org/
    │       ├── public/         
    │       └── shared/
    │           └── bob@example.org/
    │               └── files/
    │                   └── message.txt         ← ENCRYPTED message for Bob ✅
```
and if you open the encrypted `message.txt`, you will see something like this:


The encrypted message will look something like this:
```
SYC1{
  "canon": "jcs-rfc8785",
  "cipher": {
    "ciphertext_len": 26,
    "nonce": "pdBX-6oGtt8P-3cUNVc7KtazWV0kvylk",
    "suite": "xchacha20poly1305-v1"
  },
  "recipients": [{
    "identity": "bob@example.org",
    "device_label": "default"
  }],
  "sender": {
    "identity": "alice@example.org"
  },
  "wrappings": [{
    "wrap_ciphertext": "beOx7_ktlVDvr5JNmZzsw86lytCSPddav8fYn_sirRHHh4kz_BmxUCzRMWjy...[2KB base64 data]...",
    "wrap_ephemeral_public": "BfFnmJYsTntbtRc2R-M4zq5DIyw3TSR3zMsspqnVgHVb"
  }]
}@=%nD�cH��)+0R�*C��Ea6�q��\h����55z���>��]�u����T*q�_��c>�5�jx�Cz��G�d<�Y�G��	���
```

The message consists of:
- **SYC1** header followed by JSON metadata
- **Encrypted wrapper** containing ~2KB of base64-encoded ciphertext
- **Binary payload** after the `@=` delimiter

Deliver Ciphertext to Bob

In [ ]:
!cp ../sandbox/alice/datasites/alice@example.org/shared/bob@example.org/files/message.txt ../sandbox/bob/datasites/bob@example.org/shared/alice@example.org/files/message.txt

```
bob/
  ├── .syc/                                          
  ├── datasites/                                     
  │   ├── alice@example.org/                        
  │   └── bob@example.org/
  │       ├── public/                               
  │       └── shared/alice@example.org/files/
  │           └── message.txt                       ← ✅ Encrypted message FROM Alice
```

## Step 5: Bob Inspects & Decrypts the Message

First, Bob inspects the encrypted message from Alice using `syc file inspects` 

In [ ]:
!./syc file inspect --vault ../sandbox/bob/.syc --input bob@example.org/shared/alice@example.org/files/message.txt --identity bob@example.org --verbose

In [ ]:
!cat ../sandbox/bob/datasites/bob@example.org/shared/alice@example.org/files/message.txt

Now, Bob decrypts the encrypted messsage from Alice with his private keys

In [ ]:
!./syc file decrypt --vault ../sandbox/bob/.syc --relative bob@example.org/shared/alice@example.org/files/message.txt --identity bob@example.org

The decrypted plaintext resides at `sandbox/bob/unencrypted/bob@example.org/shared/alice@example.org/files/message.txt`:
```
bob/
├── .syc/
├── datasites/
└── unencrypted/
    └── bob@example.org/
        ├── public/
        └── shared/alice@example.org/files/
            └── message.txt                       ← ✅ Decrypted message FROM Alice
```

The `message.txt` will contain the `Hello Bob` message from Alice

In [ ]:
!cat ../sandbox/bob/unencrypted/bob@example.org/shared/alice@example.org/files/message.txt

---

## ✅ Success!

If you see **"Hello World"** above, syft-crypto-core is working correctly!

### What just happened?

You successfully:
1. ✅ Generated post-quantum secure keys
2. ✅ Encrypted data with PQXDH protocol
3. ✅ Decrypted and verified the data

---

## Next Steps

Continue to **02-advanced-features.ipynb** to explore:
- 🔓 Multi-recipient encryption (share with multiple people)
- 📁 File encryption with compression
- 🔐 Security features (sender exclusion, plausible deniability)
- 🔄 Key recovery system

## Cleanup (Optional)

Remove the sandboxed test environment to keep your workspace clean.

In [ ]:
!just clean-sandbox